In [ ]:
#!pip install nibabel numpy matplotlib scikit-image
#if you dont already have these in your current environment!

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path
from skimage import filters

In [ ]:
#Percentile clipping
INPUT_PATH  = '/path/to/input.nii.gz'
OUTPUT_DIR  = '/path/to/output'

# ── params ────────────────────────────────────────────────────────────────────────────────
LO_PCT    = 0.1
HI_PCT    = 99.9
SHOW_MIPS = True    # set False to skip plots
# ─────────────────────────────────────────────────────────────────────────────

def norm_percentile(vol, lo_pct, hi_pct):
    lo = np.percentile(vol, lo_pct)
    hi = np.percentile(vol, hi_pct)
    return np.clip((vol - lo) / (hi - lo), 0, 1).astype(np.float32) if hi > lo else np.zeros_like(vol, dtype=np.float32)

def show_mips(vol, title):
    total = vol.shape[0]
    depths = [max(1, int(total * 0.1)), max(1, int(total * 0.5))]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, n in zip(axes, depths):
        proj = np.max(vol[:n], axis=0)
        lo, hi = proj.min(), proj.max()
        proj = (proj - lo) / (hi - lo) if hi > lo else proj
        proj = proj[::-1]
        ax.imshow(proj, cmap='gray')
        ax.set_title(f'MIP {round(n/total*100)}% ({n} slices)', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

nii    = nib.load(INPUT_PATH)
vol    = np.transpose(nii.get_fdata().astype(np.float32), (2, 1, 0))
print(f'Loaded: {vol.shape}  min={vol.min():.2f}  max={vol.max():.2f}')

result = norm_percentile(vol, LO_PCT, HI_PCT)
print(f'After clip: min={result.min():.4f}  max={result.max():.4f}')

if SHOW_MIPS:
    show_mips(result, f'Percentile clip {LO_PCT}–{HI_PCT}%  |  {Path(INPUT_PATH).name}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
stem     = Path(INPUT_PATH).name.replace('.nii.gz', '').replace('.nii', '')
out_path = os.path.join(OUTPUT_DIR, f'{stem}_pct{LO_PCT}_{HI_PCT}.nii.gz')
nib.save(nib.Nifti1Image(np.transpose(result, (2, 1, 0)), nii.affine, nii.header), out_path)
print(f'Saved → {out_path}')

In [ ]:
#Sato filtering 2D (takes less teim then 3d)


INPUT_PATH  = '/path/to/input.nii.gz'
OUTPUT_DIR  = '/path/to/output'

# ── params ────────────────────────────────────────────────────────────────────────────────
SIGMAS    = [2, 4] # this is best for detecting large and small vessels, you can also just enter in one if you want 
BORDER    = None #ignore this for now 
SHOW_MIPS = True #set this to true if oyu want to visualize 
# ─────────────────────────────────────────────────────────────────────────────

def norm_minmax(img):
    lo, hi = img.min(), img.max()
    return (img - lo) / (hi - lo) if hi > lo else np.zeros_like(img, dtype=np.float32)

def sato_no_border(sl, sigmas, border=None):
    border = border or int(max(sigmas) * 3)
    out = filters.sato(sl, sigmas=sigmas, black_ridges=False, mode='constant', cval=0)
    out[:border, :]  = 0
    out[-border:, :] = 0
    out[:, :border]  = 0
    out[:, -border:] = 0
    return norm_minmax(out)

def show_mips(vol, title):
    total = vol.shape[0]
    depths = [max(1, int(total * 0.1)), max(1, int(total * 0.5))]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, n in zip(axes, depths):
        proj = np.max(vol[:n], axis=0)
        lo, hi = proj.min(), proj.max()
        proj = (proj - lo) / (hi - lo) if hi > lo else proj
        proj = proj[::-1]
        ax.imshow(proj, cmap='gray')
        ax.set_title(f'MIP {round(n/total*100)}% ({n} slices)', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

nii    = nib.load(INPUT_PATH)
vol    = np.transpose(nii.get_fdata().astype(np.float32), (2, 1, 0))
print(f'Loaded: {vol.shape}  min={vol.min():.4f}  max={vol.max():.4f}')

result = np.stack([sato_no_border(sl, SIGMAS, BORDER) for sl in vol]).astype(np.float32)
print(f'After Sato 2D: min={result.min():.4f}  max={result.max():.4f}')

if SHOW_MIPS:
    show_mips(result, f'Sato 2D σ={SIGMAS}  |  {Path(INPUT_PATH).name}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
stem     = Path(INPUT_PATH).name.replace('.nii.gz', '').replace('.nii', '')
out_path = os.path.join(OUTPUT_DIR, f'{stem}_sato2d_s{SIGMAS}.nii.gz')
nib.save(nib.Nifti1Image(np.transpose(result, (2, 1, 0)), nii.affine, nii.header), out_path)
print(f'Saved → {out_path}')

In [ ]:
#Sato 2D filtering with thresholding, only apply the filter to a specific intensity region


INPUT_PATH  = '/path/to/input.nii.gz'
OUTPUT_DIR  = '/path/to/output'

# ── params ────────────────────────────────────────────────────────────────────────────────
LO_INTENSITY = 1500 #sets everything below this threshold to background 
HI_INTENSITY = 5000   # set to None for no upper cap
SIGMAS       = [2, 4]
BORDER       = None
SHOW_MIPS    = True
# ─────────────────────────────────────────────────────────────────────────────

def norm_minmax(img):
    lo, hi = img.min(), img.max()
    return (img - lo) / (hi - lo) if hi > lo else np.zeros_like(img, dtype=np.float32)

def window_norm(sl, lo, hi=None):
    sl = np.clip(sl.astype(np.float32) - lo, 0, None)
    if hi is not None:
        sl = np.clip(sl / (hi - lo), 0, 1)
    else:
        mx = sl.max()
        sl = (sl / mx).astype(np.float32) if mx > 0 else sl
    return sl

def sato_no_border(sl, sigmas, border=None):
    border = border or int(max(sigmas) * 3)
    out = filters.sato(sl, sigmas=sigmas, black_ridges=False, mode='constant', cval=0)
    out[:border, :]  = 0
    out[-border:, :] = 0
    out[:, :border]  = 0
    out[:, -border:] = 0
    return norm_minmax(out)

def show_mips(vol, title):
    total = vol.shape[0]
    depths = [max(1, int(total * 0.1)), max(1, int(total * 0.5))]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, n in zip(axes, depths):
        proj = np.max(vol[:n], axis=0)
        lo, hi = proj.min(), proj.max()
        proj = (proj - lo) / (hi - lo) if hi > lo else proj
        proj = proj[::-1]
        ax.imshow(proj, cmap='gray')
        ax.set_title(f'MIP {round(n/total*100)}% ({n} slices)', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

nii      = nib.load(INPUT_PATH)
vol      = np.transpose(nii.get_fdata().astype(np.float32), (2, 1, 0))
print(f'Loaded: {vol.shape}  min={vol.min():.2f}  max={vol.max():.2f}')

windowed = np.stack([window_norm(sl, LO_INTENSITY, HI_INTENSITY) for sl in vol])
result   = np.stack([sato_no_border(sl, SIGMAS, BORDER) for sl in windowed]).astype(np.float32)
print(f'After threshold→Sato 2D: min={result.min():.4f}  max={result.max():.4f}')

if SHOW_MIPS:
    hi_tag = str(HI_INTENSITY) if HI_INTENSITY else 'uncapped'
    show_mips(result, f'Threshold {LO_INTENSITY}–{hi_tag} → Sato 2D σ={SIGMAS}  |  {Path(INPUT_PATH).name}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
stem     = Path(INPUT_PATH).name.replace('.nii.gz', '').replace('.nii', '')
hi_tag   = str(HI_INTENSITY) if HI_INTENSITY else 'uncapped'
out_path = os.path.join(OUTPUT_DIR, f'{stem}_thresh{LO_INTENSITY}_{hi_tag}_sato2d_s{SIGMAS}.nii.gz')
nib.save(nib.Nifti1Image(np.transpose(result, (2, 1, 0)), nii.affine, nii.header), out_path)
print(f'Saved → {out_path}')

In [ ]:
#3D sato filtering (takes more time)


INPUT_PATH  = '/path/to/input.nii.gz'
OUTPUT_DIR  = '/path/to/output'

# ── params ────────────────────────────────────────────────────────────────────────────────
SIGMAS    = [2, 4]
SHOW_MIPS = True
# ─────────────────────────────────────────────────────────────────────────────

def norm_minmax(vol):
    lo, hi = vol.min(), vol.max()
    return (vol - lo) / (hi - lo) if hi > lo else np.zeros_like(vol, dtype=np.float32)

def show_mips(vol, title):
    total = vol.shape[0]
    depths = [max(1, int(total * 0.1)), max(1, int(total * 0.5))]
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    for ax, n in zip(axes, depths):
        proj = np.max(vol[:n], axis=0)
        lo, hi = proj.min(), proj.max()
        proj = (proj - lo) / (hi - lo) if hi > lo else proj
        proj = proj[::-1]
        ax.imshow(proj, cmap='gray')
        ax.set_title(f'MIP {round(n/total*100)}% ({n} slices)', fontsize=11)
        ax.axis('off')
    fig.suptitle(title, fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

nii    = nib.load(INPUT_PATH)
vol    = np.transpose(nii.get_fdata().astype(np.float32), (2, 1, 0))
print(f'Loaded: {vol.shape}  min={vol.min():.4f}  max={vol.max():.4f}')

print(f'Running 3D Sato σ={SIGMAS}...', end=' ', flush=True)
result = norm_minmax(
    filters.sato(vol, sigmas=SIGMAS, black_ridges=False, mode='constant', cval=0)
).astype(np.float32)
print('done')
print(f'After Sato 3D: min={result.min():.4f}  max={result.max():.4f}')

if SHOW_MIPS:
    show_mips(result, f'Sato 3D σ={SIGMAS}  |  {Path(INPUT_PATH).name}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
stem     = Path(INPUT_PATH).name.replace('.nii.gz', '').replace('.nii', '')
out_path = os.path.join(OUTPUT_DIR, f'{stem}_sato3d_s{SIGMAS}.nii.gz')
nib.save(nib.Nifti1Image(np.transpose(result, (2, 1, 0)), nii.affine, nii.header), out_path)
print(f'Saved → {out_path}')